# 0 air temperature find out the latest 30 years

In [ ]:
# https://weatherspark.com/y/127039/Average-Weather-in-Xiantao-China-Year-Round

In [21]:
work_space= r'D:\shanghai workshop\lst\original'

# 01 landsat data process and HR

In [32]:
import glob
import os

tif_files = glob.glob(os.path.join(work_space,r'\*.tif'))

for file_path in tif_files:
    print("📂 读取 TIF 文件:", file_path)
    # 这里写读取代码


# 删除黑色区域的数据

In [33]:
import glob

tif_files = glob.glob(os.path.join(work_space,r'*QA_PIXEL.tif'))

for file_path in tif_files:
    print("📂 读取 TIF 文件:", file_path)
    # 这里写读取代码

In [34]:
import os
import glob
import rasterio
import numpy as np

tif_files = glob.glob(os.path.join(work_space,r'*QA_PIXEL.tif'))

for file_path in tif_files:
    with rasterio.open(file_path) as src:
        data = src.read(1)  # 读取第一个波段
        profile = src.profile

    # 设置 nodata 值
    nodata_val = 0
    profile['nodata'] = nodata_val

    # 将值为 1 的像素设为 nodata
    data_cleaned = np.where(data == 1, nodata_val, data)

    # 输出路径，添加 `_clean` 后缀
    dirname, basename = os.path.split(file_path)
    name, ext = os.path.splitext(basename)
    out_folder = os.path.join(dirname,r'clean')
    os.makedirs(out_folder,exist_ok=True)
    out_path = os.path.join(out_folder, f"{name}_clean{ext}")

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(data_cleaned, 1)

    print(f"✅ 已保存清理后的文件: {out_path}")


# merge

In [36]:
import os
import glob
import rasterio
from rasterio.merge import merge
import re

def extract_date_from_filename(filename):
    """从文件名中提取第 4 个下划线后的 8 位日期"""
    parts = os.path.basename(filename).split('_')
    if len(parts) > 4 and re.match(r'\d{8}', parts[3]):
        return parts[3]
    return None

def get_all_dates(folder, suffix="_clean.TIF"):
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))
    date_set = set()
    for file in tif_files:
        parts = os.path.basename(file).split('_')
        if len(parts) > 4 and re.match(r'\d{8}', parts[3]):
            date_set.add(parts[3])  # 确保只取第 4 个字段
    return sorted(list(date_set))

def merge_clean_tifs_by_date(folder, date_str, output_name_prefix="LC08_merged", suffix="_clean.TIF"):
    matching_files = []
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))

    for f in tif_files:
        parts = os.path.basename(f).split('_')
        if len(parts) > 4 and parts[3] == date_str:
            matching_files.append(f)

    if not matching_files:
        print(f"❌ 没有找到严格匹配日期 {date_str} 的文件")
        return

    print(f"\n📅 合并日期：{date_str}  ▶️ 文件数：{len(matching_files)}")
    for f in matching_files:
        print("   📄", os.path.basename(f))

    src_files = [rasterio.open(f) for f in matching_files]
    mosaic, out_trans = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": src_files[0].nodata,
        "driver": "GTiff"
    })

    output_folder = os.path.join(folder, 'merge')
    os.makedirs(output_folder, exist_ok=True)

    output_path = os.path.join(output_folder, f"{output_name_prefix}_{date_str}{suffix}")
    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    print(f"✅ 合并完成: {output_path}")


def merge_all_dates(folder, suffix="_clean.TIF"):
    all_dates = get_all_dates(folder, suffix)
    if not all_dates:
        print("⚠️ 没有找到任何带日期的匹配文件")
        return

    print(f"🔎 共发现 {len(all_dates)} 个日期: {all_dates}")
    for date_str in all_dates:
        merge_clean_tifs_by_date(folder, date_str, suffix=suffix)


merge_all_dates(os.path.join(work_space, 'clean'), suffix="_QA_PIXEL_clean.TIF")


🔎 共发现 144 个日期: ['20150702', '20150709', '20150718', '20150725', '20150803', '20150810', '20150819', '20150826', '20160602', '20160609', '20160618', '20160625', '20160704', '20160711', '20160720', '20160727', '20160805', '20160812', '20160821', '20160828', '20170605', '20170612', '20170621', '20170628', '20170707', '20170714', '20170723', '20170730', '20170808', '20170815', '20170824', '20170831', '20180608', '20180615', '20180624', '20180701', '20180710', '20180717', '20180726', '20180802', '20180811', '20180818', '20180827', '20190602', '20190611', '20190618', '20190627', '20190704', '20190713', '20190720', '20190729', '20190805', '20190814', '20190821', '20190830', '20200604', '20200613', '20200620', '20200629', '20200706', '20200715', '20200722', '20200731', '20200807', '20200816', '20200823', '20210607', '20210616', '20210623', '20210702', '20210709', '20210718', '20210725', '20210803', '20210810', '20210819', '20210826', '20220602', '20220603', '20220610', '20220611', '20220618', 

In [38]:
import os
import glob
import rasterio
from rasterio.merge import merge

def extract_date_from_filename(filename):
    parts = os.path.basename(filename).split('_')
    if len(parts) > 4:
        return parts[3]  # 第 4 个字段是日期
    return None

def get_all_dates(folder, suffix="_ST_B10.TIF"):
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))
    date_set = set()
    for file in tif_files:
        date_str = extract_date_from_filename(file)
        if date_str:
            date_set.add(date_str)
    return sorted(list(date_set))

def merge_clean_tifs_by_date(folder, date_str, output_name_prefix="LC08_merged", suffix="_ST_B10.TIF"):
    matching_files = []
    tif_files = glob.glob(os.path.join(folder, f"*{suffix}"))

    for f in tif_files:
        parts = os.path.basename(f).split('_')
        if len(parts) > 4 and parts[3] == date_str:
            matching_files.append(f)

    if not matching_files:
        print(f"❌ 没有找到匹配日期 {date_str} 的文件")
        return

    print(f"\n📅 合并日期：{date_str}  ▶️ 文件数：{len(matching_files)}")
    for f in matching_files:
        print("   📄", os.path.basename(f))

    src_files = [rasterio.open(f) for f in matching_files]
    mosaic, out_trans = merge(src_files)

    out_meta = src_files[0].meta.copy()
    out_meta.update({
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_trans,
        "nodata": src_files[0].nodata,
        "driver": "GTiff"
    })

    output_folder = os.path.join(folder,'clean', 'merge')
    os.makedirs(output_folder, exist_ok=True)

    # 安全提取 band 名（比如 "_ST_B10.TIF" -> "ST_B10"）
    band_part = suffix.strip("_").replace(".TIF", "")
    output_filename = f"{output_name_prefix}_{date_str}_{band_part}.TIF"

    output_path = os.path.join(output_folder, output_filename)

    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(mosaic)

    print(f"✅ 合并完成: {output_path}")

def merge_all_dates(folder, suffix="_ST_B10.TIF"):
    all_dates = get_all_dates(folder, suffix)
    if not all_dates:
        print("⚠️ 没有找到任何带日期的文件")
        return

    print(f"🔎 共发现 {len(all_dates)} 个日期: {all_dates}")
    for date_str in all_dates:
        merge_clean_tifs_by_date(folder, date_str, suffix=suffix)

# ✅ 调用
merge_all_dates(work_space, suffix="_ST_B10.TIF")


🔎 共发现 144 个日期: ['20150702', '20150709', '20150718', '20150725', '20150803', '20150810', '20150819', '20150826', '20160602', '20160609', '20160618', '20160625', '20160704', '20160711', '20160720', '20160727', '20160805', '20160812', '20160821', '20160828', '20170605', '20170612', '20170621', '20170628', '20170707', '20170714', '20170723', '20170730', '20170808', '20170815', '20170824', '20170831', '20180608', '20180615', '20180624', '20180701', '20180710', '20180717', '20180726', '20180802', '20180811', '20180818', '20180827', '20190602', '20190611', '20190618', '20190627', '20190704', '20190713', '20190720', '20190729', '20190805', '20190814', '20190821', '20190830', '20200604', '20200613', '20200620', '20200629', '20200706', '20200715', '20200722', '20200731', '20200807', '20200816', '20200823', '20210607', '20210616', '20210623', '20210702', '20210709', '20210718', '20210725', '20210803', '20210810', '20210819', '20210826', '20220602', '20220603', '20220610', '20220611', '20220618', 

# clip by the boundary

In [45]:
import os
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from shapely.geometry import box
from glob import glob

def clip_rasters_with_shapefile(raster_folder, shapefile_path, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    # Load the shapefile as a GeoDataFrame
    polygon = gpd.read_file(shapefile_path)

    for raster_path in glob(os.path.join(raster_folder, '*.tif')):
        with rasterio.open(raster_path) as src:
            raster_crs = src.crs

            # Reproject polygon to raster CRS if needed
            if polygon.crs != raster_crs:
                polygon_proj = polygon.to_crs(raster_crs)
            else:
                polygon_proj = polygon

            # Raster bounds as a Shapely box
            raster_bounds = box(*src.bounds)

            # Polygon total bounds (merged geometry)
            polygon_union = polygon_proj.unary_union
            polygon_bounds = polygon_union.bounds
            polygon_box = box(*polygon_bounds)

            # Only clip if raster fully contains the polygon's bounding box
            if not raster_bounds.contains(polygon_box):
                print(f"❌ 跳过（raster 不包含 polygon）: {os.path.basename(raster_path)}")
                continue

            # Clip raster
            out_image, out_transform = mask(src, polygon_proj.geometry, crop=True)

            # Update metadata
            out_meta = src.meta.copy()
            out_meta.update({
                "driver": "GTiff",
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            output_raster_path = os.path.join(output_folder, f"clipped_{os.path.basename(raster_path)}")
            with rasterio.open(output_raster_path, "w", **out_meta) as dest:
                dest.write(out_image)

            print(f"✅ 已裁切并保存: {output_raster_path}")

# 示例调用
clip_rasters_with_shapefile(
    raster_folder=r"D:\shanghai workshop\lst\original\clean\merge",
    shapefile_path=r"D:\shanghai workshop\vector\上海市_Polygon.shp",
    output_folder=r"D:\shanghai workshop\lst\original\clean\merge_clipped"
)

C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_QA_PIXEL_clean.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_ST_B10.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20150709_QA_PIXEL_clean.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20150709_ST_B10.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


...

✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240819_ST_B10.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20240826_QA_PIXEL_clean.TIF
❌ 跳过（raster 不包含 polygon）: LC08_merged_20240826_ST_B10.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240827_QA_PIXEL_clean.TIF


C:\Users\owner\AppData\Local\Temp\ipykernel_16756\37876436.py:28: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  polygon_union = polygon_proj.unary_union


✅ 已裁切并保存: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20240827_ST_B10.TIF


## 2 计算百分比

In [46]:
import rasterio
import numpy as np
import os
import re

def get_sorted_dates(file_path, suffix="QA_PIXEL_clean.TIF"):
    file_names = os.listdir(file_path)
    date_keys = []

    for fname in file_names:
        if fname.endswith(suffix):
            match = re.search(r'(\d{8})', fname)
            if match:
                date_keys.append(match.group(1))

    sorted_dates = sorted(set(date_keys))
    print("📅 提取的日期：", sorted_dates)
    return sorted_dates

def calculate_pixel_proportion(sorted_dates, base_folder, pixel_values=[21824, 21952]):
    for date_str in sorted_dates:
        # 找到匹配这个日期的文件（不拼死结构）
        matched_files = [f for f in os.listdir(base_folder)
                         if f.endswith("QA_PIXEL_clean.TIF") and date_str in f]

        if not matched_files:
            print(f"⚠️ 没有找到包含日期 {date_str} 的 QA 文件")
            continue

        file_path = os.path.join(base_folder, matched_files[0])
        print(f"📂 正在分析: {file_path}")

        with rasterio.open(file_path) as src:
            qa_pixel_data = src.read(1)

        non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
        total_non_zero_pixels = non_zero_pixels.size

        if total_non_zero_pixels == 0:
            print(f"⚠️ 所有像素为0: {file_path}")
            continue

        unique, counts = np.unique(non_zero_pixels, return_counts=True)
        pixel_counts = dict(zip(unique, counts))

        pixel_proportions = {val: pixel_counts.get(val, 0) / total_non_zero_pixels for val in pixel_values}

        print(f"📅 {date_str}:")
        for val in pixel_values:
            prop = pixel_proportions.get(val, 0)
            print(f"  - 像素值 {val} 的占比: {prop:.2%}")
        print()

# 示例路径
base_folder = os.path.join(work_space,'clean','merge_clipped')
sorted_dates = get_sorted_dates(base_folder)
calculate_pixel_proportion(sorted_dates, base_folder)

📅 提取的日期： ['20150702', '20150718', '20150803', '20150819', '20160602', '20160618', '20160704', '20160720', '20160805', '20160821', '20170605', '20170621', '20170707', '20170723', '20170808', '20170824', '20180608', '20180624', '20180710', '20180726', '20180811', '20180827', '20190611', '20190627', '20190713', '20190729', '20190814', '20190830', '20200613', '20200629', '20200715', '20200731', '20200816', '20210616', '20210702', '20210718', '20210803', '20210819', '20220603', '20220611', '20220619', '20220627', '20220705', '20220713', '20220721', '20220729', '20220806', '20220814', '20220822', '20220830', '20230606', '20230614', '20230622', '20230630', '20230708', '20230716', '20230724', '20230801', '20230809', '20230817', '20230825', '20240608', '20240616', '20240624', '20240702', '20240710', '20240718', '20240726', '20240803', '20240811', '20240819', '20240827']
📂 正在分析: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150702_QA_PIXEL_clean.TIF
📅 20150702:
  - 

In [50]:
import rasterio
import numpy as np

def find_high_proportion_dates(sorted_dates, input_folder, threshold=0.80, target_value=[21824,21952] ):
    high_proportion_dates = []

    for date in sorted_dates:
        # Path to the QA_PIXEL TIFF file for the current date
        data_path = rf'{input_folder}\clipped_LC08_merged_{date}_QA_PIXEL_clean.tif'

        try:
            with rasterio.open(data_path) as src:
                qa_pixel_data = src.read(1)  # Read the first band data

            # Filter for non-zero pixels
            non_zero_pixels = qa_pixel_data[qa_pixel_data > 0]
            total_non_zero_pixels = non_zero_pixels.size

            # Count unique pixel values and calculate proportions
            unique, counts = np.unique(non_zero_pixels, return_counts=True)
            pixel_counts = dict(zip(unique, counts))
            pixel_proportions = {value: count / total_non_zero_pixels for value, count in pixel_counts.items()}

            # Check the proportion of the target values
            proportion_target_value = sum(pixel_proportions.get(val, 0) for val in target_value)
            if proportion_target_value > threshold:
                high_proportion_dates.append((date, proportion_target_value))
                print(f"{date} pixel values {target_value} combined proportion: {proportion_target_value:.2%}")

        except FileNotFoundError:
            print(f"File not found: {data_path}")

    return high_proportion_dates

input_folder= os.path.join(work_space,'clean','merge_clipped')
high_proportion_dates = find_high_proportion_dates(sorted_dates, input_folder)

# Print results
print("\nDates with high target value proportion:")
for date, proportion in high_proportion_dates:
    print(f"{date}: {proportion:.2%}")

20150803 pixel values [21824, 21952] combined proportion: 97.98%
20150819 pixel values [21824, 21952] combined proportion: 83.20%
20160720 pixel values [21824, 21952] combined proportion: 95.34%
20170723 pixel values [21824, 21952] combined proportion: 95.61%
20170824 pixel values [21824, 21952] combined proportion: 99.61%
20180726 pixel values [21824, 21952] combined proportion: 85.85%
20190729 pixel values [21824, 21952] combined proportion: 97.08%
20190814 pixel values [21824, 21952] combined proportion: 89.73%
20190830 pixel values [21824, 21952] combined proportion: 83.54%
20200816 pixel values [21824, 21952] combined proportion: 99.98%
20220814 pixel values [21824, 21952] combined proportion: 99.67%
20220822 pixel values [21824, 21952] combined proportion: 93.13%
20240803 pixel values [21824, 21952] combined proportion: 98.29%
20240811 pixel values [21824, 21952] combined proportion: 89.01%

Dates with high target value proportion:
20150803: 97.98%
20150819: 83.20%
20160720: 95.3

20150803: 97.98% 28-38
20150819: 83.20% 26-33
20160720: 95.34% 29-36
20170723: 95.61% 30-40
20170824: 99.61% 28-36
20180726: 85.85% 28-36
20190729: 97.08% 29-36
20190814: 89.73% 27-34
20190830: 83.54% 22-30
20200816: 99.98% 28-37
20220814: 99.67% 29-39
20220822: 93.13% 30-39
20240803: 98.29% 29-39
20240811: 89.01% 28-38

# 20170723: 95.61%

# 用 QA数据清理B10 -> clean data

In [52]:
import rasterio
import numpy as np
import os
for date, proportion in high_proportion_dates:
    data_workspace = os.path.join(work_space,r"clean\merge_clipped")
    b_10_data = os.path.join(data_workspace, rf'clipped_LC08_merged_{date}_ST_B10.tif')
    qa_data = os.path.join(data_workspace, rf'clipped_LC08_merged_{date}_QA_PIXEL_clean.tif')
    output_folder = os.path.join(data_workspace,r'clean')
    os.makedirs(output_folder,exist_ok=True)

    output_data = os.path.join(output_folder, rf'clipped_{date}_B10_cleaned.tif')
    print("B10 Data Path:", b_10_data)
    print("QA Data Path:", qa_data)
    print("Output Data Path:", output_data)

    # Read Band 10 and QA data
    with rasterio.open(b_10_data) as src:
        band10 = src.read(1).astype(float)  # Convert data to float type to support NaN
        profile = src.profile

    with rasterio.open(qa_data) as qa_src:
        qa = qa_src.read(1)  # Read QA pixel data

    # Define labels for different cloud and shadow conditions
    label = [1, 21824, 21826, 21888, 21890, 22080, 22144, 22280, 23888, 23952, 24088, 24216, 24344, 24472, 30048, 54596, 54852, 55052]
    print("Dilated cloud over land QA label:", label[2])

    # Create a mask for clouds and shadows, filtering out cloud and shadow areas
    valid_labels = [21824, 21952]
    cloud_mask = ~np.isin(qa, valid_labels)  # mask 出非有效值（云、阴影、水等）

    # Set cloud and shadow areas to 0
    band10[cloud_mask] = 0

    # Save the processed result
    with rasterio.open(output_data, 'w', **profile) as dst:
        dst.write(band10, 1)

B10 Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150803_ST_B10.tif
QA Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150803_QA_PIXEL_clean.tif
Output Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clean\clipped_20150803_B10_cleaned.tif
Dilated cloud over land QA label: 21826
B10 Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150819_ST_B10.tif
QA Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20150819_QA_PIXEL_clean.tif
Output Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clean\clipped_20150819_B10_cleaned.tif
Dilated cloud over land QA label: 21826
B10 Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20160720_ST_B10.tif
QA Data Path: D:\shanghai workshop\lst\original\clean\merge_clipped\clipped_LC08_merged_20160720_QA_PIXEL_clean.tif
Output Data Path: D:\sh

# 先将数据洗成摄氏度 bt -> lst

In [23]:
import rasterio
import numpy as np
import os

def calculate_lst(input_files, output_dir, multiplier=0.00341802, add_constant=149.0, nodata_value=-9999):
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    for input_file in input_files:
        with rasterio.open(input_file) as src:
            # Read the first band and convert to float
            band_data = src.read(1).astype(float)
            profile = src.profile

        # Replace values <= 0 with NaN
        band_data[band_data <= 0] = np.nan

        # Calculate brightness temperature (BT)
        lst_kelvin = (band_data * multiplier) + add_constant

        # Convert to Celsius
        lst_celsius = lst_kelvin - 273.15

        # Replace NaN values with the nodata value
        lst_celsius[np.isnan(lst_celsius)] = nodata_value

        # Update profile for the output file
        profile.update({
            'dtype': 'float32',
            'count': 1,
            'nodata': nodata_value
        })

        # Generate a short output file name
        base_name = os.path.basename(input_file).split('.')[0]
        output_file = os.path.join(output_dir, f"{base_name}_LST.tif")

        # Save the LST raster
        with rasterio.open(output_file, 'w', **profile) as dst:
            dst.write(lst_celsius, 1)

        print(f"LST saved to: {output_file}")


# 输入的栅格文件路径
landsat_files = [
    # os.path.join(work_space,r'clean\merge_clipped\clean\clipped_20170723_B10_cleaned.tif'),
    # os.path.join(work_space,r'clean\merge_clipped\clean\clipped_20220814_B10_cleaned.tif'),
    os.path.join(work_space,r'clean\merge_clipped\clean\clipped_20200816_B10_cleaned.tif')
]

# 输出目录
output_dir = os.path.join(work_space,r'final_lst')
os.makedirs(output_dir,exist_ok=True)

# 计算并保存 LST
calculate_lst(landsat_files, output_dir)

LST saved to: D:\shanghai workshop\lst\original\final_lst\clipped_20200816_B10_cleaned_LST.tif


# vector

In [63]:

import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"lst_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(

    original_raster=r'D:\shanghai workshop\lst\original\final_lst\clipped_20220814_B10_cleaned_LST.tif',
    grid_path=r'D:\shanghai workshop\vector\case.shp',
    output_dir=r"D:\shanghai workshop\raster\lst_case"
)

📌 原始栅格坐标系: EPSG:32651
🔄 正在将矢量网格投影为与栅格一致的坐标系...
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54143.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54144.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54145.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54146.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54147.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54486.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54487.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54488.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54489.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54490.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54829.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54830.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54831.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_case\lst_tile_54832.tif
✅ 已保存切块: D:\shanghai workshop\raster\lst_ca

In [62]:
import geopandas as gpd

grid_path = r'D:\shanghai workshop\vector\case.shp'
gdf = gpd.read_file(grid_path)

# 打印字段名
print("字段名：", gdf.columns)

# 如果你想单独看 'ID' 列的值，可以这样：
if 'ID' in gdf.columns:
    print("ID 列的前几项：")
    print(gdf['ID'])
else:
    print("⚠️ 没有找到 'ID' 这一列，请检查字段名是否正确。")

字段名： Index(['ID', 'FID_1', 'geometry'], dtype='object')
ID 列的前几项：
0     54143
1     54144
2     54145
3     54146
4     54147
5     54486
6     54487
7     54488
8     54489
9     54490
10    54829
11    54830
12    54831
13    54832
14    54833
15    55172
16    55173
17    55174
18    55175
19    55176
20    55515
21    55516
22    55517
23    55518
24    55519
Name: ID, dtype: int64


In [64]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"lst_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(

    original_raster=r'D:\shanghai workshop\lst\original\final_lst\clipped_20220814_B10_cleaned_LST.tif',
    grid_path=r'D:\shanghai workshop\vector\case_merge.shp',
    output_dir=r"D:\shanghai workshop\raster\case_merge"
)

📌 原始栅格坐标系: EPSG:32651
🔄 正在将矢量网格投影为与栅格一致的坐标系...
✅ 已保存切块: D:\shanghai workshop\raster\case_merge\lst_tile_54143.tif


# water

In [4]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_fclass_by_grid(shapefile_path, grid_path, fclass_mapping, output_dir, resolution):
    # 读取水体数据和网格
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统一为 UTM 51N
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 保留指定类别的记录，并转换为数值
    gdf = gdf[gdf['fclass'].isin(fclass_mapping.keys())]
    gdf['class_val'] = gdf['fclass'].map(fclass_mapping)

    # 清理空 geometry
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    for _, cell in grid.iterrows():
        tile_id = cell["ID"]  # 使用 ID 字段作为输出文件名
        grid_geom = cell.geometry

        # 裁剪出当前格子内的水体
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")

        if clipped.empty:
            continue

        # 计算输出 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 构造 shapes
        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=255,  # Nodata 为 255
            dtype='uint8'
        )

        out_path = os.path.join(output_dir, f"water_tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=255
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 示例调用
fclass_mapping = {
    'dock': 0,
    'reservoir': 1,
    'riverbank': 2,
    'water': 3,
    'wetland': 4
}

rasterize_fclass_by_grid(
    shapefile_path=r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp",
    grid_path=rf"D:\shanghai workshop\vector\case_merge.shp",
    fclass_mapping=fclass_mapping,
    output_dir=r"D:\shanghai workshop\raster\case_merge",
    resolution=2
)

✅ 输出: D:\shanghai workshop\raster\case_merge\water_tile_54143.tif


In [5]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
import os

def split_raster_by_grid(original_raster, grid_path, output_dir):
    # 读取矢量网格
    grid = gpd.read_file(grid_path)

    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    with rasterio.open(original_raster) as src:
        raster_crs = src.crs
        print("📌 原始栅格坐标系:", raster_crs)

        # 坐标系统一
        if grid.crs != raster_crs:
            print("🔄 正在将矢量网格投影为与栅格一致的坐标系...")
            grid = grid.to_crs(raster_crs)

        for _, row in grid.iterrows():
            tile_id = row["ID"]  # 使用 grid 中的 ID 字段
            geom = [row.geometry]

            try:
                out_image, out_transform = mask(src, geom, crop=True)
            except Exception as e:
                print(f"⚠️ 跳过 ID={tile_id}，裁剪失败: {e}")
                continue

            out_meta = src.meta.copy()
            out_meta.update({
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "crs": raster_crs
            })

            out_path = os.path.join(output_dir, f"building_tile_{tile_id}.tif")
            with rasterio.open(out_path, "w", **out_meta) as dst:
                dst.write(out_image)

            print(f"✅ 已保存切块: {out_path}")

# 示例调用
split_raster_by_grid(
    original_raster=r'D:\shanghai workshop\raster\building_32651.tif',
    grid_path=r"D:\shanghai workshop\vector\case_merge.shp",
    output_dir=r"D:\shanghai workshop\raster\case_merge"
)

📌 原始栅格坐标系: EPSG:32651
🔄 正在将矢量网格投影为与栅格一致的坐标系...
✅ 已保存切块: D:\shanghai workshop\raster\case_merge\building_tile_54143.tif


# street

In [6]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_by_grid_with_auto_class_mapping(shapefile_path, grid_path, class_field, output_dir, resolution=10, nodata=255):
    # 读取数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统统一为投影坐标系（UTM 51N）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 清理无效几何
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    # 检查字段是否存在
    if class_field not in gdf.columns:
        raise ValueError(f"字段 {class_field} 不存在于 shapefile 中")

    # 获取字段的唯一值列表，并生成映射字典（类别字符串->数字ID，从1开始，0 留作背景）
    unique_classes = sorted(gdf[class_field].dropna().unique())
    class_to_id = {cls: idx+1 for idx, cls in enumerate(unique_classes)}
    print(f"自动生成类别映射（{class_field}）：{class_to_id}")

    # 用映射替换原字段，未包含的赋值0
    gdf['class_val'] = gdf[class_field].map(class_to_id).fillna(0).astype(int)

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=nodata,
            dtype='uint8'
        )
        tile_id = cell["ID"]
        out_path = os.path.join(output_dir, f"tile_{tile_id}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 使用示例
rasterize_by_grid_with_auto_class_mapping(
    shapefile_path=r"D:\shanghai workshop\vector\Shanghai street_ver3\Shanghai street_ver3.shp",
    grid_path=r"D:\shanghai workshop\vector\\case_merge.shp",
    class_field="Type",
    output_dir=r"D:\shanghai workshop\raster\\case_merge",
    resolution=1,
    nodata=255
)

自动生成类别映射（Type）：{'中心城区主要道路': 1, '中心城区道路': 2, '主干道路': 3, '公路': 4, '快速路': 5, '快速路匝道': 6, '支路': 7, '郊县中心城区主干道路': 8, '高速': 9, '高速公路匝道': 10}
✅ 输出: D:\shanghai workshop\raster\\case_merge\tile_54143.tif


In [18]:
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform, Resampling as WarpResampling
import numpy as np
import os

def reproject_and_resample(src_path, ref_crs, ref_transform, ref_width, ref_height):
    try:
        with rasterio.open(src_path) as src:
            dst_array = np.empty((ref_height, ref_width), dtype=np.float32)

            reproject(
                source=rasterio.band(src, 1),
                destination=dst_array,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                resampling=WarpResampling.nearest
            )
            return dst_array
    except Exception as e:
        print(f"❌ 无法读取或重采样 {src_path}：{e}")
        return None

def resample_and_merge_multiband(base_raster, layers_dict, output_path):
    with rasterio.open(base_raster) as ref:
        src_crs = ref.crs
        dst_crs = rasterio.crs.CRS.from_epsg(32651)

        dst_transform, dst_width, dst_height = calculate_default_transform(
            src_crs, dst_crs, ref.width, ref.height, *ref.bounds
        )

        ref_meta = ref.meta.copy()

    dtype = np.float32
    nodata_val = np.nan
    layer_order = ["water", "park", "street", "building","landuse","lst"]
    band_count = len(layer_order)

    # 初始化为 NaN
    merged_array = np.full((band_count, dst_height, dst_width), nodata_val, dtype=dtype)

    for i, layer in enumerate(layer_order):
        path = layers_dict.get(layer)
        if path and os.path.exists(path):
            print(f"📥 正在处理图层 {layer}: {path}")
            arr = reproject_and_resample(path, dst_crs, dst_transform, dst_width, dst_height)

            if arr is not None:
                valid_mask = (~np.isnan(arr)) & (arr != 0) & (arr != 255)
                cleaned = np.full_like(arr, nodata_val)
                cleaned[valid_mask] = arr[valid_mask]
                merged_array[i] = cleaned
            else:
                print(f"⚠️ 图层 {layer} 重采样失败，保留为 NaN")
        else:
            print(f"⚠️ 图层 {layer} 缺失或路径无效：{path}")

    out_meta = ref_meta.copy()
    out_meta.update({
        "count": band_count,
        "dtype": "float32",
        "height": dst_height,
        "width": dst_width,
        "transform": dst_transform,
        "crs": dst_crs,
        "nodata": nodata_val
    })

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with rasterio.open(output_path, "w", **out_meta) as dst:
        dst.write(merged_array)
        print(f"✅ 多波段 GeoTIFF 已保存: {output_path}")

# 示例调用
n = 54143
base_raster = rf"D:\shanghai workshop\raster\case_merge\tile_{n}.tif"
layers = {
    "building": rf"D:\shanghai workshop\raster\case_merge\building_tile_{n}.tif",
    "park": rf"D:\shanghai workshop\raster\case_merge\park_tile_{n}.tif",
    "water": rf"D:\shanghai workshop\raster\case_merge\water_tile_{n}.tif",
    "street": rf"D:\shanghai workshop\raster\case_merge\tile_{n}.tif",
    "landuse": rf"D:\shanghai workshop\raster\final_landuse_tiles\landuse_tile_{n}.tif",
    "lst": rf"D:\shanghai workshop\raster\lst\lst_tile_{n}.tif"

}
output_path = rf"D:\shanghai workshop\raster\merged_multiband\merged_tile_{n}.tif"
resample_and_merge_multiband(base_raster, layers, output_path)


📥 正在处理图层 water: D:\shanghai workshop\raster\case_merge\water_tile_54143.tif
⚠️ 图层 park 缺失或路径无效：D:\shanghai workshop\raster\case_merge\park_tile_54143.tif
📥 正在处理图层 street: D:\shanghai workshop\raster\case_merge\tile_54143.tif
📥 正在处理图层 building: D:\shanghai workshop\raster\case_merge\building_tile_54143.tif
📥 正在处理图层 landuse: D:\shanghai workshop\raster\final_landuse_tiles\landuse_tile_54143.tif
📥 正在处理图层 lst: D:\shanghai workshop\raster\lst\lst_tile_54143.tif
✅ 多波段 GeoTIFF 已保存: D:\shanghai workshop\raster\merged_multiband\merged_tile_54143.tif


In [19]:
import os
import geopandas as gpd
import rasterio
from rasterio.mask import mask

# 读取矢量网格
vector_path = r"D:\shanghai workshop\vector\case_merge.shp"
gdf = gpd.read_file(vector_path)

# 确保 CRS 和栅格一致（即 EPSG:32651）
gdf = gdf.to_crs(epsg=32651)

# 获取所有 ID 列表
id_list = gdf['ID'].tolist()
id_list = [54143]
# 遍历每个 ID，裁剪同名栅格
for n in id_list:
    input_raster = rf"D:\shanghai workshop\raster\merged_multiband\merged_tile_{n}.tif"
    output_raster = rf"D:\shanghai workshop\raster\merged_multiband\merged_clean_tile_{n}.tif"

    if not os.path.exists(input_raster):
        print(f"⚠️ 跳过缺失文件: merged_tile_{n}.tif")
        continue

    try:
        # 提取该 ID 对应的 polygon
        polygon = gdf[gdf['ID'] == n].geometry.values[0]
        geojson_geom = [polygon.__geo_interface__]

        # 打开栅格并裁剪
        with rasterio.open(input_raster) as src:
            out_image, out_transform = mask(src, geojson_geom, crop=True, nodata=np.nan)
            out_meta = src.meta.copy()

        # 更新元信息
        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": np.nan
        })

        # 写出裁剪后的文件
        os.makedirs(os.path.dirname(output_raster), exist_ok=True)
        with rasterio.open(output_raster, "w", **out_meta) as dest:
            dest.write(out_image)
            print(f"✅ 裁剪完成: clipped_tile_{n}.tif")

    except Exception as e:
        print(f"❌ 裁剪 tile_{n} 出错: {e}")

✅ 裁剪完成: clipped_tile_54143.tif


In [24]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os
import numpy as np

def rasterize_multiple_fields_to_multiband(shapefile_path, grid_path, class_fields, output_dir, resolution=10, nodata=255, start_id=0):
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    # ✅ 第一步：提前为所有字段生成全局映射字典
    field_maps = {}
    for field in class_fields:
        if field not in gdf.columns:
            raise ValueError(f"字段 {field} 不存在于 shapefile 中")
        if gdf[field].dtype.kind in ['O', 'U']:  # 字符型
            unique_vals = sorted(gdf[field].dropna().unique())
            val_map = {v: i+1 for i, v in enumerate(unique_vals)}
            print(f"字段 {field} 的映射：{val_map}")
            field_maps[field] = val_map
            gdf[f"__mapped_{field}"] = gdf[field].map(val_map).fillna(0).astype(int)
        else:
            gdf[f"__mapped_{field}"] = gdf[field].fillna(0)

    for idx, cell in grid.iterrows():
        tile_id = cell["ID"]

        # ✅ 跳过起始 ID 之前的 tile
        if tile_id < start_id:
            continue

        out_path = os.path.join(output_dir, f"building_tile_{tile_id}.tif")

        if os.path.exists(out_path):
            print(f"⏩ 已存在，跳过: {out_path}")
            continue

        grid_geom = cell.geometry
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        bands = []

        for field in class_fields:
            mapped_col = f"__mapped_{field}"
            class_vals = clipped[mapped_col]
            shapes = [(geom, val) for geom, val in zip(clipped.geometry, class_vals)]

            raster = rasterize(
                shapes=shapes,
                out_shape=(height, width),
                transform=transform,
                fill=nodata,
                dtype='uint8' if class_vals.dtype == int else 'float32'
            )
            bands.append(raster)

        bands_array = np.stack(bands, axis=0)

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=len(class_fields),
            dtype=bands_array.dtype,
            crs=gdf.crs,
            transform=transform,
            nodata=nodata
        ) as dst:
            dst.write(bands_array)

        print(f"✅ 输出: {out_path}")

rasterize_multiple_fields_to_multiband(
    shapefile_path=r'D:\shanghai workshop\vector\Building_processed nearest 1 to 500m_finver\result\building_UTM51N.shp',
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_256m_clipped.shp",
    class_fields=["Height", "Function", "Style"],
    output_dir=r"D:\shanghai workshop\raster\final_building_tiles_with_bands",
    resolution=1,
    nodata=255,
    start_id=60000
)

字段 Function 的映射：{'Business': 1, 'Industry': 2, 'Office': 3, 'Other': 4, 'Public service': 5, 'Residence': 6}
字段 Style 的映射：{'M': 1, 'Modern': 2, 'Postmodern': 3, 'Traditional': 4}
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60161.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60165.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60168.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60170.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60171.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60172.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60173.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60174.tif
✅ 输出: D:\shanghai workshop\raster\final_building_tiles_with_bands\building_tile_60175.tif
✅ 输出: D:\sh